In [2]:
import pandas as pd
import re

df = pd.read_excel("train_fixed.xlsx")
print(df.head())

   review_id                                        review_text  star_rating  \
0       7238                لا يوجد الدفع بالبطاقه عند الاستلام            3   
1       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...            5   
2       1975  تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...            1   
3       3024                                    احلي مكان فزايد            5   
4       5483  الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...            4   

                  date              business_name      business_category  \
0  2026-03-08 00:00:00                       Noon              ecommerce   
1        قبل يومين (2)      ممشي مصر Mawlana Cafe                  كافيه   
2              قبل شهر          بيت لحم Beet Lahm                   مطعم   
3              قبل شهر  ذا بلكون كافيه الشيخ زايد  مطعم مأكولات ومشروبات   
4              قبل سنة        The Best Restaurant                   مطعم   

      platform                                 aspects  \
0   

In [3]:
print(df.columns)

Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='object')


In [4]:
# Convert JSON strings into real objects
import json

df["aspects"] = df["aspects"].apply(json.loads)
df["aspect_sentiments"] = df["aspect_sentiments"].apply(json.loads)

In [5]:
# Convert ABSA into ML format
rows = []

for _, row in df.iterrows():
    review = row["review_text"]
    sentiments = row["aspect_sentiments"]

    for aspect, sentiment in sentiments.items():
        rows.append({
            "text": review,
            "aspect": aspect,
            "label": sentiment
        })

train_df = pd.DataFrame(rows)

print(train_df.head())

                                                text          aspect     label
0                لا يوجد الدفع بالبطاقه عند الاستلام  app_experience  negative
1                لا يوجد الدفع بالبطاقه عند الاستلام        delivery  negative
2  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...     cleanliness  positive
3  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...        ambiance  positive
4  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...         service  positive


In [6]:
# Convert labels to numbers
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

train_df["label_id"] = train_df["label"].map(label_map)

In [8]:
train_df["input"] = train_df["text"] + " " + train_df["aspect"]

In [9]:
# Tokenization
from datasets import Dataset

dataset = Dataset.from_pandas(train_df)

def tokenize(batch):
    return tokenizer(
        batch["input_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

dataset = dataset.map(tokenize)

ModuleNotFoundError: No module named 'pyarrow'

In [7]:
# Load Arabic BERT model
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "aubmindlab/bert-base-arabertv2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

c:\Users\dalia\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dalia\.cache\huggingface\hub\models--aubmindlab--bert-base-arabertv2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
